# 01. Scaled Dot-Product Attention — 밑바닥부터

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glasslego/2026-ml-study/blob/main/notebooks/study_03_transformer/01_attention_from_scratch.ipynb)

**목표**: "Attention is All You Need" (Vaswani et al., 2017) 의 핵심 연산인 *scaled dot-product attention* 을
수식 → numpy 직접 구현 → PyTorch 검증 → 시각화 순으로 이해한다.

흐름:
1. 동기 — RNN 한계, attention 의 등장 이유
2. Q, K, V 의 직관 — "사전 검색" 비유
3. 수식: $\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\dfrac{QK^\top}{\sqrt{d_k}}\right) V$
4. numpy 로 직접 구현
5. PyTorch `F.scaled_dot_product_attention` 와 결과 일치 검증
6. Causal mask (decoder/언어모델용)
7. Attention weight 시각화
8. Self-attention vs Cross-attention

> 이 노트북을 끝내면 "GPT/BERT 가 안에서 뭘 하는지" 의 80% 가 보입니다. 나머지 20% (multi-head, positional encoding,
> residual, layer norm, FFN) 는 다음 노트북 `02_multihead_and_transformer_block.ipynb` 에서 다룹니다.


In [ ]:
# 공통 실행 환경 준비 (Colab/Local 통합)
from pathlib import Path
import subprocess, sys, os

REPO_URL = 'https://github.com/glasslego/2026-ml-study.git'
REPO_NAME = '2026-ml-study'
NOTEBOOK_SUBDIR = 'notebooks/study_03_transformer'

if 'google.colab' in sys.modules:
    clone_root = Path('/content') / REPO_NAME
    if not clone_root.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(clone_root)], check=True)
    target = clone_root / NOTEBOOK_SUBDIR
else:
    target = None
    for c in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (c / NOTEBOOK_SUBDIR).exists():
            target = c / NOTEBOOK_SUBDIR
            break
    if target is None:
        raise FileNotFoundError(f'{NOTEBOOK_SUBDIR} 를 찾지 못했습니다.')

os.chdir(target)
print(f'작업 디렉토리: {target}')

try:
    import torch  # noqa
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch'], check=True)
    import torch  # noqa


## 1. 왜 Attention 인가 — RNN 의 한계

이전 노트북(`study_02_cnn_rnn`) 에서 본 RNN/LSTM 의 두 가지 약점:

1. **순차 처리** — timestep $t$ 의 hidden 을 계산하려면 $t-1$ 의 hidden 이 있어야 한다. GPU 병렬화 불리.
2. **긴 의존성** — "문장 첫 단어" 의 정보가 $h_T$ 까지 전달되려면 $T$ 번의 행렬 곱을 거쳐야 한다.
   gradient 가 vanish 하기 쉽다.

Attention 의 발상은 단순합니다:

> 현재 위치가 **모든** 다른 위치를 "직접" 들여다보고, 자신과 관련 깊은 곳에 가중치를 더 주자.

이러면:
- 모든 위치가 병렬로 처리된다 (행렬 한 번에).
- 어떤 두 위치 사이의 거리가 얼마든 "한 hop" 으로 정보 교환 가능 (gradient 경로가 짧다).


## 2. Q, K, V 의 직관 — "사전 검색" 비유

각 입력 토큰이 세 가지 표현을 만든다:

- **Query ($q$)** : "내가 찾고 싶은 정보"
- **Key ($k$)**   : "내가 가진 정보의 색인(label)"
- **Value ($v$)** : "내가 실제로 줄 정보"

비유: 도서관에서 책을 찾을 때 — 검색어(Q) 와 각 책의 제목(K) 의 유사도를 보고, 가장 닮은 책의 내용(V) 을 가져온다.
attention 은 이걸 "hard" 가 아니라 **"soft" (가중 평균)** 으로 한다.

수식 한 토큰 $i$ 에 대해:

$$ \text{output}_i = \sum_j \underbrace{\text{softmax}_j\!\left(\frac{q_i \cdot k_j}{\sqrt{d_k}}\right)}_{\text{토큰 i 가 토큰 j 에 주는 attention 가중치}} \cdot v_j $$

이걸 모든 $i$ 에 대해 한 번에 행렬로 쓰면:

$$ \boxed{\;\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V\;} $$

여기서 $Q \in \mathbb{R}^{N \times d_k}$, $K \in \mathbb{R}^{M \times d_k}$, $V \in \mathbb{R}^{M \times d_v}$.

**왜 $\sqrt{d_k}$ 로 나누나?** — $d_k$ 가 크면 $q \cdot k$ 의 분산이 커지고, softmax 의 입력이 커져서
gradient 가 거의 0인 영역(saturated region)으로 빠진다. 이를 방지하기 위한 스케일링.


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

np.random.seed(0)
torch.manual_seed(0)


## 3. numpy 로 scaled dot-product attention 직접 구현

수식을 그대로 코드로 옮긴다. `softmax_axis_last` 만 주의하면 어렵지 않다.


In [ ]:
def softmax_np(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """수치 안정성을 위해 max 를 빼고 softmax 를 계산한다.

    softmax(x_i) = exp(x_i) / sum_j exp(x_j)
    그대로 계산하면 큰 x 에서 exp overflow. max 를 빼도 결과는 동일 (수학적 항등식).
    """
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)


def attention_np(Q: np.ndarray, K: np.ndarray, V: np.ndarray,
                 mask: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray]:
    """Scaled Dot-Product Attention.

    Args:
        Q: (..., N, d_k) — 쿼리 N개
        K: (..., M, d_k) — 키 M개
        V: (..., M, d_v) — 값 M개
        mask: (..., N, M) bool. True 인 위치는 attend 금지 (예: causal, padding mask).

    Returns:
        out:    (..., N, d_v)
        weights:(..., N, M) — softmax 후의 attention 가중치 (해석/시각화에 유용)
    """
    d_k = Q.shape[-1]
    # 1) 점곱: scores[..., i, j] = Q[..., i, :] · K[..., j, :]
    scores = Q @ np.swapaxes(K, -1, -2)         # (..., N, M)
    # 2) 스케일: 큰 d_k 에서 softmax 포화 방지
    scores = scores / np.sqrt(d_k)
    # 3) 마스킹: 금지 위치는 -inf 로 → softmax 후 0
    if mask is not None:
        scores = np.where(mask, -1e9, scores)
    # 4) softmax 로 가중치 정규화
    weights = softmax_np(scores, axis=-1)        # 행 합 = 1
    # 5) 가중 평균
    out = weights @ V                            # (..., N, d_v)
    return out, weights


# 작은 예: N=3 쿼리, M=4 키/값, d_k=d_v=8
N, M, d_k, d_v = 3, 4, 8, 8
Q = np.random.randn(N, d_k).astype(np.float32)
K = np.random.randn(M, d_k).astype(np.float32)
V = np.random.randn(M, d_v).astype(np.float32)

out_np, w_np = attention_np(Q, K, V)
print('output shape :', out_np.shape, '  (N, d_v)')
print('weights shape:', w_np.shape, '  (N, M) — 각 행 합이 1')
print('각 행의 합   :', w_np.sum(axis=-1))


## 4. PyTorch 결과와 일치하는지 검증

PyTorch 2.0 부터 `F.scaled_dot_product_attention` 이 내장(고도로 최적화된 FlashAttention 백엔드 포함). 같은 입력에서
우리 numpy 구현과 결과가 일치해야 한다.


In [ ]:
Qt = torch.from_numpy(Q)
Kt = torch.from_numpy(K)
Vt = torch.from_numpy(V)

# PyTorch 의 SDPA. 배치 차원을 요구하므로 unsqueeze.
out_torch = F.scaled_dot_product_attention(
    Qt.unsqueeze(0), Kt.unsqueeze(0), Vt.unsqueeze(0)
).squeeze(0).numpy()

print('일치 여부 :', np.allclose(out_np, out_torch, atol=1e-6))
print('최대 오차 :', np.max(np.abs(out_np - out_torch)))


## 5. Causal Mask — "미래를 못 보게 막기"

언어모델(decoder) 에서는 토큰 $i$ 가 미래 토큰 $j > i$ 를 보면 **정답 누설** 이 된다.
그래서 위쪽(미래) 위치에 마스크를 씌워 attention 가중치를 0으로 만든다.

마스크 행렬 (N=M=4 예시, True = 차단):

$$
\text{mask} =
\begin{pmatrix}
F & T & T & T \\
F & F & T & T \\
F & F & F & T \\
F & F & F & F
\end{pmatrix}
$$

대각선 위쪽(상삼각) 이 모두 차단되는 형태다.


In [ ]:
T = 4
Q2 = np.random.randn(T, d_k).astype(np.float32)
K2 = np.random.randn(T, d_k).astype(np.float32)
V2 = np.random.randn(T, d_v).astype(np.float32)

# upper triangular (k=1: 대각선 자체는 보존, 그 위만 차단)
causal_mask = np.triu(np.ones((T, T), dtype=bool), k=1)
print('Causal mask:')
print(causal_mask.astype(int))

out_c, w_c = attention_np(Q2, K2, V2, mask=causal_mask)
print('\n각 행의 가중치 (위쪽이 0이어야 함):')
print(np.round(w_c, 3))


## 6. Attention Weight 시각화

가중치 행렬 자체가 "누가 누구를 본다" 의 해석 가능한 정보다.
무거운 의존성을 피하기 위해 matplotlib 만 사용한다.


In [ ]:
import matplotlib.pyplot as plt

# 같은 마스크된 가중치를 히트맵으로
fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(w_c, cmap='Blues', vmin=0, vmax=1)
for i in range(T):
    for j in range(T):
        ax.text(j, i, f'{w_c[i,j]:.2f}', ha='center', va='center',
                color='white' if w_c[i,j] > 0.5 else 'black', fontsize=9)
ax.set_xlabel('key index (j)')
ax.set_ylabel('query index (i)')
ax.set_title('Causal Attention Weights')
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()


## 7. Self-attention vs Cross-attention

같은 연산 식이지만, **무엇을 Q/K/V 로 넣느냐** 에 따라 두 가지 용법이 갈린다.

| 종류              | Q 의 출처    | K, V 의 출처   | 쓰임                                          |
|-------------------|--------------|----------------|-----------------------------------------------|
| **Self-attention**| 같은 시퀀스  | **같은** 시퀀스| BERT/GPT 의 모든 층, 시퀀스 안에서 토큰끼리 "대화" |
| **Cross-attention**| 디코더       | **인코더 출력** | 번역 모델: 디코더가 인코더의 표현을 참조           |

수식은 동일하다. 차이는 입력 텐서의 출처일 뿐.


In [ ]:
# (a) Self-attention: 같은 시퀀스에서 Q=K=V_source 로 시작 (선형 변환은 multi-head 에서 다룸)
seq = np.random.randn(5, d_k).astype(np.float32)   # 5개 토큰의 임베딩
out_self, _ = attention_np(seq, seq, seq)
print('self-attention 출력 모양 :', out_self.shape, '  (5, d_v)')

# (b) Cross-attention: Q 는 디코더, K/V 는 인코더 출력
dec_seq = np.random.randn(3, d_k).astype(np.float32)   # 디코더 3 토큰
enc_seq = np.random.randn(7, d_k).astype(np.float32)   # 인코더 7 토큰
out_cross, _ = attention_np(dec_seq, enc_seq, enc_seq)
print('cross-attention 출력 모양:', out_cross.shape, '  (3, d_v)  — Q 의 길이 (3)')


## 8. 흔한 함정 (Common Pitfalls)

1. **스케일링 누락**: $\sqrt{d_k}$ 로 안 나누면 $d_k$ 가 클수록 softmax 가 거의 one-hot 으로 포화 → gradient 0.
2. **mask 부호**: PyTorch `attn_mask` 는 "True 가 차단" 이지만 `additive mask` 형식은 "-inf 가 차단, 0 이 허용". 라이브러리마다 다르니 docstring 확인.
3. **softmax axis**: 마지막 축(=key 축) 에서 정규화해야 한다. 잘못 잡으면 의미가 깨진다.
4. **padding mask**: 가변 길이 배치에서 padding 토큰 위치를 안 막으면 "의미 없는 위치" 에 attention 이 새어 나간다.
5. **dtype**: float16 에서 -1e9 마스크 값은 -inf 로 처리되지 않을 수 있음. `-torch.finfo(dtype).max` 또는 `float('-inf')` 사용.


## 정리

- 한 줄: $\mathrm{Attention}(Q, K, V) = \mathrm{softmax}(QK^\top/\sqrt{d_k})\, V$
- 직관: 쿼리가 키와의 유사도로 가중치를 만들어 값의 가중 평균을 가져온다.
- 마스크로 "못 보는 위치" (causal/padding) 를 막는다.
- self vs cross 는 입력 출처의 차이일 뿐 연산은 동일.

다음 노트북 `02_multihead_and_transformer_block.ipynb` 에서:
- 여러 head 가 다른 부분공간을 동시에 보는 **multi-head** 확장
- 순서 정보를 주입하는 **positional encoding**
- Layer Norm, FFN, Residual 까지 합친 **풀 transformer encoder/decoder block**
